# Notebook 1 — Phase A: Data Building (vLLM batched) — **Kaggle version**

Produces `sft_data_L1.jsonl`, `sft_data_L2.jsonl`, `dpo_pairs.jsonl` in **`/kaggle/working/P-ALIGN/data`**.

**Before running on Kaggle:**
- Settings → **Accelerator = GPU T4 x2** (hoặc P100)
- Settings → **Internet = On** (cần để `pip install` + tải model/dataset từ HuggingFace)

**Generation strategy:** round-based batching with vLLM.
- Round 1: all 966 questions sent to vLLM in one call
- Score results → ~20% trigger prefix feedback
- Round 2 & 3: only feedback rows resent

**Resumable:** skip-if-exists on every output file (chỉ resume trong cùng 1 session — xem ghi chú lưu file ở cuối).


In [ ]:
# Cell 1 — Kaggle setup & install dependencies
# (KHÔNG mount Google Drive trên Kaggle)
!pip install -q transformers accelerate tqdm datasets math-verify sympy peft vllm

import os

# Tắt FlashInfer sampler: tránh JIT-compile kernel (lỗi `cannot find -lcuda` trên Kaggle).
# vLLM sẽ fallback sang top-k/top-p sampler native của PyTorch.
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# /kaggle/working  -> được LƯU LẠI và xuất ra Output (tải về / tạo Dataset được)
BASE_DIR = "/kaggle/working/P-ALIGN"
os.makedirs(f"{BASE_DIR}/data", exist_ok=True)

# Model tải về /kaggle/temp (scratch) để KHÔNG chiếm quota 20GB của Output
MODEL_BASE = "/kaggle/temp/models"
os.makedirs(MODEL_BASE, exist_ok=True)

print(f"BASE_DIR (outputs): {BASE_DIR}")
print(f"MODEL_BASE (scratch): {MODEL_BASE}")


In [ ]:
# Cell 2 — Model: dùng Kaggle Dataset nếu đã attach, nếu không thì tải từ HuggingFace
from huggingface_hub import snapshot_download

# Nếu bạn đã thêm model dưới dạng Kaggle Dataset, sửa path input bên dưới cho khớp.
KAGGLE_MODEL_INPUT = "/kaggle/input/qwen2.5-1.5b-instruct"

if os.path.exists(f"{KAGGLE_MODEL_INPUT}/config.json"):
    MODEL_DIR = KAGGLE_MODEL_INPUT
    print(f"[Kaggle dataset] Using model at {MODEL_DIR}")
else:
    # Tải về scratch (cần Internet = On). KHÔNG vào Output -> không tốn quota.
    MODEL_DIR = f"{MODEL_BASE}/Qwen2.5-1.5B-Instruct"
    if not os.path.exists(f"{MODEL_DIR}/config.json"):
        snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=MODEL_DIR)
        print(f"Downloaded → {MODEL_DIR}")
    else:
        print(f"[Resume] Model already at {MODEL_DIR}")


In [ ]:
# Cell 3 — Load & Join Datasets → prefixes.jsonl
import re, json, unicodedata
from datasets import load_dataset

PREFIX_FILE = f"{BASE_DIR}/data/prefixes.jsonl"

if os.path.exists(PREFIX_FILE):
    with open(PREFIX_FILE) as f:
        rows = [json.loads(l) for l in f]
    print(f"[Resume] Loaded {len(rows)} rows from {PREFIX_FILE}")
else:
    # ── Step A: parse P-ALIGN (Alpaca format) ─────────────────────────────
    # instruction = system prompt (same for all rows, ignore it)
    # input       = math question  ← join key
    # output      = prefix, tagged with <Begin_of_Prefix>  ← strip tag
    palign = load_dataset("qizheyanger/P-ALIGN", split="train")
    print(f"P-ALIGN columns: {palign.column_names}")
    print(f"Sample input:  {repr(palign[0]['input'][:150])}")
    print(f"Sample output: {repr(palign[0]['output'][:150])}")

    _PREFIX_TAG = re.compile(r"<Begin_of_Prefix>\s*", re.IGNORECASE)

    palign_parsed = []
    for item in palign:
        question = item["input"].strip()
        prefix   = _PREFIX_TAG.sub("", item["output"]).strip()
        palign_parsed.append({"question": question, "prefix": prefix})

    print(f"\nParsed {len(palign_parsed)} P-ALIGN rows")

    # ── Step B: load s1K-1.1 ──────────────────────────────────────────────
    s1k = load_dataset("simplescaling/s1K-1.1", split="train")
    print(f"s1K-1.1 columns: {s1k.column_names}")

    def _norm(text: str) -> str:
        text = unicodedata.normalize("NFC", text)
        return re.sub(r"\s+", " ", text.lower().strip())

    s1k_lookup      = {}
    s1k_lookup_norm = {}
    for item in s1k:
        key = item["question"].strip()
        val = {
            "full_cot": item["deepseek_thinking_trajectory"],
            "answer":   str(item["solution"]).strip(),
        }
        s1k_lookup[key]             = val
        s1k_lookup_norm[_norm(key)] = val
    print(f"s1K-1.1 lookup: {len(s1k_lookup)} entries")

    # ── Step C: join on question text (exact → normalized → 120-char prefix) ─
    rows, n_miss = [], 0
    for i, parsed in enumerate(palign_parsed):
        q = parsed["question"]

        lookup = s1k_lookup.get(q)
        if lookup is None:
            lookup = s1k_lookup_norm.get(_norm(q))
        if lookup is None:
            q_fp = _norm(q)[:120]
            lookup = next(
                (v for k, v in s1k_lookup_norm.items() if k[:120] == q_fp),
                None,
            )
        if lookup is None:
            n_miss += 1
            continue

        rows.append({
            "id":       i,
            "question": q,
            "prefix":   parsed["prefix"],
            "answer":   lookup["answer"],
            "full_cot": lookup["full_cot"],
        })

    print(f"\nJoined {len(rows)} samples ({n_miss} unmatched, dropped)")
    if len(rows) == 0:
        raise RuntimeError(
            "Join still produced 0 rows.\n"
            f"Sample P-ALIGN question: {repr(palign_parsed[0]['question'][:200])}\n"
            f"Sample s1K key:          {repr(list(s1k_lookup.keys())[0][:200])}"
        )

    with open(PREFIX_FILE, "w") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved → {PREFIX_FILE}")

In [ ]:
# Cell 4 — Load vLLM Engine + Tokenizer
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"  # phải set TRƯỚC khi tạo LLM(...)

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# Tokenizer — only for apply_chat_template (no model weights loaded here)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# vLLM engine: PagedAttention + continuous batching
# T4 has 15 GB VRAM; model ~3 GB → 80% utilization leaves ~9 GB for KV cache
llm = LLM(
    model=MODEL_DIR,
    dtype="float16",
    gpu_memory_utilization=0.80,
    max_model_len=4096,        # prompt (≤2048) + generation (≤2048)
    trust_remote_code=True,
)
print("vLLM engine ready.")


In [ ]:
# Cell 5 — Hyperparameters
K          = 4
TEMP       = 0.8
TOP_P      = 0.95
MAX_NEW    = 2048
TAU_LOW    = 0.2
T_MAX      = 3
LAMBDA_LEN = 0.0
EPSILON    = 1e-6
CLIP_C     = 3.0

# vLLM SamplingParams: n=K → K outputs per prompt in one shot
SAMPLING_PARAMS = SamplingParams(
    n=K,
    temperature=TEMP,
    top_p=TOP_P,
    max_tokens=MAX_NEW,
)
print(f"K={K}  MAX_NEW={MAX_NEW}  TEMP={TEMP}")

In [ ]:
# Cell 6 — Prompt Builder
def build_formatted_prompt(question: str, prefix: str) -> str:
    """Return a chat-template string ready for vLLM input."""
    content = (
        "Please continue from the draft and solve the problem step by step, "
        "and put your final answer within \\boxed{}. "
        "I will provide you with some prior knowledge as a draft to assist you.\n"
        f"*Question*: {question}\n"
        f"*Prefix*: {prefix}"
    )
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

In [ ]:
# Cell 7 — Batch Generator
def batch_generate(prompts: list) -> list:
    """
    Send N prompts to vLLM, get K continuations per prompt.
    Returns list[list[str]], shape [N, K].
    All N prompts are processed concurrently via PagedAttention.
    """
    outputs = llm.generate(prompts, SAMPLING_PARAMS)
    return [[o.text for o in req.outputs] for req in outputs]

In [ ]:
# Cell 8 — Verifier (Step 4)
import re, signal, math
from math_verify import parse, verify

def extract_boxed(text: str) -> str:
    matches = re.findall(r"\\boxed\{([^}]*)\}", text)
    return matches[-1] if matches else ""

def verify_answer(pred: str, gold: str, timeout_sec: int = 5) -> int:
    def _handler(s, f): raise TimeoutError
    old = signal.signal(signal.SIGALRM, _handler)
    signal.alarm(timeout_sec)
    try:
        pred_ans = extract_boxed(pred)
        if not pred_ans: return 0
        return int(bool(verify(parse(pred_ans), parse("$" + str(gold) + "$"))))
    except Exception:
        return 0
    finally:
        signal.alarm(0); signal.signal(signal.SIGALRM, old)

def compute_rewards(continuations: list, answer: str) -> list:
    rewards = []
    for y in continuations:
        r = float(verify_answer(y, answer))
        if LAMBDA_LEN > 0:
            r -= LAMBDA_LEN * len(y.split())
        rewards.append(r)
    return rewards

In [ ]:
# Cell 9 — Group-Relative Advantage (Step 5)
def compute_advantage(rewards: list) -> list:
    mu       = sum(rewards) / len(rewards)
    variance = sum((r - mu) ** 2 for r in rewards) / len(rewards)
    sigma    = math.sqrt(variance) + EPSILON
    return [(r - mu) / sigma for r in rewards]

def classify_group(rewards: list) -> str:
    if all(r > 0  for r in rewards): return "all_correct"
    if all(r <= 0 for r in rewards): return "all_wrong"
    return "mixed"

In [ ]:
# Cell 10 — Prefix-Feedback Helpers (Step 6)
def sentence_split(text: str) -> list:
    parts = text.split(". ")
    return [s.strip() + ("" if s.strip().endswith(".") else ".") for s in parts if s.strip()]

def build_prefix_ladder(row: dict) -> list:
    sents = sentence_split(row.get("full_cot", ""))
    return [" ".join(sents[:i+1]) for i in range(len(sents))]

def get_next_prefix(current_prefix: str, ladder: list):
    return next((c for c in ladder if len(c) > len(current_prefix)), None)

In [ ]:
# Cell 11 — Main Data-Building Loop (round-based batching with vLLM)
import json
from tqdm import tqdm

SFT_L1_FILE = f"{BASE_DIR}/data/sft_data_L1.jsonl"
SFT_L2_FILE = f"{BASE_DIR}/data/sft_data_L2.jsonl"
DPO_FILE    = f"{BASE_DIR}/data/dpo_pairs.jsonl"
STATS_FILE  = f"{BASE_DIR}/data/build_stats.json"

# Resume: collect already-processed source IDs
processed_ids = set()
for fpath in [SFT_L1_FILE, SFT_L2_FILE]:
    if os.path.exists(fpath):
        with open(fpath) as f:
            for line in f:
                processed_ids.add(json.loads(line).get("source_id", ""))
print(f"Already processed: {len(processed_ids)} source IDs")

with open(PREFIX_FILE) as f:
    all_rows = [json.loads(l) for l in f]

# Build per-row state for every unprocessed row
row_states = {}
active = []
for row in all_rows:
    sid = str(row.get("id", row["question"][:40]))
    if sid in processed_ids:
        continue
    row_states[sid] = {
        "row":    row,
        "prefix": row["prefix"],
        "ladder": build_prefix_ladder(row),
    }
    active.append(sid)

print(f"Rows needing generation: {len(active)}")
stats   = {"total": len(active), "all_correct": 0, "all_wrong": 0,
           "mixed": 0, "feedback_triggered": 0}
results = {}   # sid → {conts, rewards, prefix}

f_l1  = open(SFT_L1_FILE, "a", encoding="utf-8")
f_l2  = open(SFT_L2_FILE, "a", encoding="utf-8")
f_dpo = open(DPO_FILE,    "a", encoding="utf-8")

def write_results(sid):
    res        = results[sid]
    state      = row_states[sid]
    row        = state["row"]
    prefix     = res["prefix"]
    conts      = res["conts"]
    rewards    = res["rewards"]
    answer     = str(row["answer"])
    pass_rate  = sum(1 for r in rewards if r > 0) / K
    group_type = classify_group(rewards)
    stats[group_type] = stats.get(group_type, 0) + 1

    if group_type == "all_wrong":
        return

    advantages = compute_advantage(rewards)
    mu_r       = sum(rewards) / len(rewards)
    sum_w_l1   = sum(1.0 for r in rewards if r >= mu_r) + EPSILON
    sum_w_l2   = sum(max(a, 0) for a in advantages) + EPSILON

    for y, r, a_k in zip(conts, rewards, advantages):
        full_seq = prefix + "\n" + y
        w_l1 = (1.0 / sum_w_l1) if r >= mu_r else 0.0
        w_l2 = min(max(a_k, 0), CLIP_C) / sum_w_l2
        base = {"source_id": sid, "question": row["question"], "prefix": prefix,
                "continuation": y, "full_sequence": full_seq,
                "answer": answer, "reward": r, "pass_rate": pass_rate}
        if w_l1 > 0:
            f_l1.write(json.dumps({**base, "weight": w_l1}, ensure_ascii=False) + "\n")
        if w_l2 > 0:
            f_l2.write(json.dumps({**base, "weight": w_l2}, ensure_ascii=False) + "\n")

    if group_type == "mixed":
        best_i  = max(range(K), key=lambda i: rewards[i])
        worst_i = min(range(K), key=lambda i: rewards[i])
        if rewards[best_i] > rewards[worst_i]:
            f_dpo.write(json.dumps({
                "source_id": sid, "question": row["question"], "prefix": prefix,
                "chosen": conts[best_i], "rejected": conts[worst_i],
                "reward_chosen": rewards[best_i], "reward_rejected": rewards[worst_i],
            }, ensure_ascii=False) + "\n")

try:
    for attempt in range(T_MAX):
        if not active:
            break
        print(f"\n[Round {attempt+1}/{T_MAX}] Sending {len(active)} prompts to vLLM ...")

        prompts = [
            build_formatted_prompt(
                row_states[sid]["row"]["question"],
                row_states[sid]["prefix"],
            )
            for sid in active
        ]

        # One batched call — vLLM schedules all prompts concurrently
        all_conts = batch_generate(prompts)   # [N, K]

        next_active = []
        for sid, conts in tqdm(zip(active, all_conts), total=len(active),
                               desc=f"  Scoring round {attempt+1}"):
            state     = row_states[sid]
            rewards   = compute_rewards(conts, str(state["row"]["answer"]))
            pass_rate = sum(1 for r in rewards if r > 0) / K

            if pass_rate < TAU_LOW and attempt < T_MAX - 1:
                next_p = get_next_prefix(state["prefix"], state["ladder"])
                if next_p:
                    state["prefix"] = next_p
                    stats["feedback_triggered"] += 1
                    next_active.append(sid)
                    continue

            results[sid] = {"conts": conts, "rewards": rewards, "prefix": state["prefix"]}

        # Write everything finalized this round
        for sid in active:
            if sid not in next_active and sid in results:
                write_results(sid)
        f_l1.flush(); f_l2.flush(); f_dpo.flush()

        print(f"  Finalized: {len(active) - len(next_active)}  |  "
              f"Feedback needed: {len(next_active)}")
        active = next_active

    # Remaining rows after T_MAX — write best available
    for sid in active:
        if sid in results:
            write_results(sid)

finally:
    f_l1.close(); f_l2.close(); f_dpo.close()

with open(STATS_FILE, "w") as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))

In [ ]:
# Cell 12 — Sanity Check & Stats
import json

for label, fpath in [("L1", SFT_L1_FILE), ("L2", SFT_L2_FILE), ("DPO", DPO_FILE)]:
    with open(fpath) as f:
        lines = f.readlines()
    print(f"{label}: {len(lines)} samples in {fpath}")

with open(STATS_FILE) as f:
    print(json.load(f))